# AIN421 Final Project  
**Author:** Nihat Guliyev  
**Number:** 2230765064    
**Course:** AIN421 · Fuzzy Logic   

---

## Executive Summary

This project compares rule-based Mamdani FIS and data-driven Takagi–Sugeno ANFIS models for predicting **student “remarks” (5-class labels)** from a 200 000-row synthetic dataset with seven numeric attributes (x1–x7).

| Stage | Model | Rules | Best ACC† | Key takeaway |
|-------|-------|------:|----------:|--------------|
| Manual FIS | **FIS-1** (x7 only) | 5 | ≈ 0.54 | x7 (exam score) dominates but misclassifies boundary cases |
| + Overlap refinement | **FIS-2 / FIS-3** | 10 | ≈ 0.58 | Hierarchical “overlap-gated” rules cut boundary errors |
| ANFIS baseline | **ANFIS-x7** | 5 | 0.52 | Training alone can’t solve 1-D ambiguity |
| ANFIS full-feature | **ANFIS-7D** | 192 | **0.993** | Seven raw features give near-perfect separation |
| ANFIS compact | **ANFIS-(S + x7)** | 15 | 0.991 | 15 × smaller than 7-D with only 0.15 pp loss |

† Mean of two IID splits (37 500 train / 12 500 test each).

### Contributions
* **Pure-NumPy implementation** of Mamdani and TS-ANFIS—no external fuzzy libraries.  
* Novel **overlap-gated refinement** strategy that activates secondary inputs only where adjacent x7 memberships overlap.  
* A **15-rule ANFIS** reaching ≈ 99 % accuracy, balancing interpretability and performance.  
* Exhaustive hyper-parameter sweeps & ablation logs; all code and results released for full reproducibility.


## 1 · Dataset & Exploratory Data Analysis (EDA)

> **All artefacts listed below live under `data/` or `results/` in the submission package.  
> No placeholder figures are introduced.**

---

### 1.1  Raw-dataset overview

| Item | Value |
|------|-------|
| **Source file** | `data/raw/academicPerformanceData.xlsx` |
| **Rows** | 200 000 |
| **Columns** | 8  (`x1 … x6`, `x7`, `remarks`) |
| **Target** | `remarks ∈ {1, 2, 3, 4, 5}` |

The header sits on Excel row 2; all following rows are purely numeric.

---

### 1.2  Integrity checks

| Check | Output file | Result |
|-------|-------------|--------|
| Column dtypes | `results/data_analysis/tables/column_dtypes.csv` | all numeric |
| Missing values | `results/data_analysis/tables/missing_values.csv` | 0 |
| Basic stats | `results/data_analysis/tables/dataset_overview.csv` | confirms 200 k × 8 |

No cleaning or imputation was required.

---

### 1.3  Class balance of **remarks** (Task 2.2)

*CSV:* `results/data_analysis/tables/remarks_distribution.csv`

| Class | Count | % of data |
|------:|------:|----------:|
| 1 | 50 327 | 25.2 % |
| 2 | 47 061 | 23.5 % |
| 3 | 47 493 | 23.7 % |
| 4 | 37 270 | 18.6 % |
| 5 | 17 849 | 8.9 % |

The skew is mild (every class ≥ 8.9 %), yet still large enough to justify **stratified** splits in Section 3 so the rarest class (5) is never under-represented in train/test.

---

### 1.4  Feature statistics (Task 2.3)

*Table:* `results/data_analysis/tables/feature_statistics.csv`

| Feature | Count | Mean | Std | Min | 25 % | Median | 75 % | Max |
|---------|------:|-----:|----:|----:|----:|-------:|----:|----:|
| x1 | 200 000 | 5.0006 | 3.1624 | 0 | 2 | 5 | 8 | 10 |
| x2 | 200 000 | 5.0004 | 3.1609 | 0 | 2 | 5 | 8 | 10 |
| x3 | 200 000 | 5.0065 | 3.1650 | 0 | 2 | 5 | 8 | 10 |
| x4 | 200 000 | 5.0009 | 3.1623 | 0 | 2 | 5 | 8 | 10 |
| x5 | 200 000 | 5.0006 | 3.1623 | 0 | 2 | 5 | 8 | 10 |
| x6 | 200 000 | 4.9995 | 3.1627 | 0 | 2 | 5 | 8 | 10 |
| x7 | 200 000 | 19.9953 | 11.8310 | 0 | 10 | 20 | 30 | 40 |

*Per-feature histograms:* `results/data_analysis/feature_distributions/plots/`

Key findings

* **x1 … x6** share near-identical Gaussian-like shapes (`μ ≈ 5`, `σ ≈ 3`) on the interval 0–10.  
* **x7** spans 0–40 with `μ ≈ 20`, `σ ≈ 12`, i.e. a different scale and a heavier tail.

---

### 1.5  Correlation matrix (Task 2.4)

*Files generated*  
* Matrix → `results/data_analysis/tables/correlation_matrix.csv`  
* Heat-map → `results/data_analysis/plots/correlation_heatmap.png`

|              | x1 | x2 | x3 | x4 | x5 | x6 | x7 | remarks |
|--------------|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:-------:|
| **x1**       |1.00|-0.00|-0.00| 0.00| -0.00|-0.00|-0.00| 0.203 |
| **x2**       |-0.00|1.00|-0.00|-0.00|-0.00|-0.00|-0.00| 0.202 |
| **x3**       |-0.00|-0.00|1.00| 0.00|-0.00|-0.00| 0.00| 0.204 |
| **x4**       | 0.00|-0.00| 0.00|1.00| 0.00|-0.00|-0.00| 0.203 |
| **x5**       |-0.00|-0.00|-0.00| 0.00|1.00| 0.00| 0.00| 0.204 |
| **x6**       |-0.00|-0.00|-0.00|-0.00| 0.00|1.00| 0.00| 0.203 |
| **x7**       |-0.00|-0.00| 0.00|-0.00| 0.00| 0.00|1.00| **0.822** |
| **remarks**  | 0.203| 0.202| 0.204| 0.203| 0.204| 0.203| **0.822** | 1.00 |

![Correlation heat-map](results/data_analysis/plots/correlation_heatmap.png)

Notable numbers  

* `corr(remarks, x7) = 0.82` — by far the strongest single relationship.  
* `corr(remarks, x1…x6) = 0.20 ± 0.02` — very weak.

---

### 1.6  Feature-vs-target plots (Task 2.5)

Directory `results/data_analysis/plots/feature_vs_remarks/`

x7 example:

![x7 by remarks](results/data_analysis/plots/feature_vs_remarks/x7_by_remarks_hist.png)

x1 example (representative of x1–x6):

![x1 by remarks](results/data_analysis/plots/feature_vs_remarks/x1_by_remarks_hist.png)

Interpretation  

* Histograms of **x1–x6** for all classes overlap almost perfectly → negligible standalone predictive power.  
* **x7** shows five clearly separated modes; empirical cut-points at ≈ 10.5, 19.5, 28, 34 guide fuzzy-set design in Section 4.

---

### 1.7  Derived aggregate feature **S**

Because x1–x6 are homogeneous, we compress them into

$$
\boxed{S = \tfrac{1}{6}\sum_{i=1}^{6} x_i}
$$

* Range 0–10 — same scale, preserves mean information.  
* Enables a 2-input model (**ANFIS-(S + x7)**) with only 15 rules while retaining the weak auxiliary signal.

---

### 1.8  EDA-driven modelling choices

| Empirical insight | Consequence for model design |
|-------------------|------------------------------|
| `ρ(remarks,x7) ≈ 0.82` | Always allocate **5 fuzzy sets** to x7 and make it the primary antecedent in every rule-base. |
| x1–x6 are weak & redundant | Merge into **S** *or* use only **2 MFs each** to keep the 7-D rule explosion manageable (192 rules). |
| Mild imbalance (class 5 ≈ 9 %) | Use **stratified train/test splits** (Section 3) so every class has equal representation. |
| x7 scale ≫ x1–x6 | Apply **Min-Max scaling** before ANFIS to equalise SGD gradient magnitudes. |
| Clean numeric data | Hand-written Python implementations (no heavy ML libraries) suffice; no preprocessing pipeline required. |

These data-driven decisions underpin every subsequent FIS/ANFIS architecture, the hyper-parameter grids we swept, and the rationale for the final model selection.


## 2&nbsp;·&nbsp;Mamdani FIS Modelling & Tuning

> All Python sources live in **`src/fis_models/`**, every numeric output in  
> **`results/fis_results/`**.  
> The section follows the chronological design–>evaluate–>refine cycle 
---

### 2.1  Why start with Mamdani systems?

| Requirement from the syllabus | How it is met |
|-------------------------------|---------------|
| **Interpretability** – every student must be able to defend each rule in front of the class. | A Mamdani FIS expresses the decision as plain IF–THEN statements.  |
| **Manual design experience** – do not rely on libraries. | I wrote a *mini-fuzzy engine* (`fis_utils.py`) handling:<br>• triangular membership; • MAX/MIN aggregation;<br>• centroid defuzzification; • vectorised NumPy for speed. |
| **Two independent 100-sample tests** (`fis_run1`, `fis_run2`) | A tiny split forces us to witness *high statistical variance* and think about robustness, not only best-case scores. |

The hand-designed systems became the *conceptual laboratory* whose successes and failures
directly guided the later ANFIS architectures.

---

### 2.2  FIS-1 — **x7-only “naked” baseline**

#### 2.2.1  Membership-function (MF) placement  
* Five triangles (VL, L, M, H, VH).  
* Peak positions chosen by *empirical quartiles* of the class-conditional histograms  
  (visualised in `x7_by_remarks_hist.png`).  
* Boundary points: **10.5 | 19.5 | 28.0 | 34.0** (see Section 1).

> **Why triangles?**  They are the simplest MF shape, computationally cheap, and
> their linear sides keep the centroid calculation analytic.

#### 2.2.2  Rule base (5 core rules)

| Rule ID | Antecedent | Class (`remarks`) |
|---------|------------|------------------------|
| R1 | *x7 is VL* | 1 |  
| R2 | *x7 is L*  | 2 |
| R3 | *x7 is M*  | 3 |  
| R4 | *x7 is H*  | 4 |  
| R3 | *x7 is VH*  | 5 |  

These 5 base rules were created from the histogram of x7 by remarks and the statistics of change of x7 values by remarks.

#### 2.2.3  First attempt at “ boundary refinements”  
I naïvely added 5 more rules such as  
> R6: **IF** x7 ≈ boundary(1 | 2) **THEN** remarks = 1

Outcome:

| Variant | run1 | run2 | What went wrong |
|---------|-----:|-----:|-----------------|
| 5 core  | **0.54** | **0.54** | — baseline |
| +5 refine | 0.53 | 0.48 | Centroid started *blending* adjacent classes – refinements pull output in both directions simultaneously |

**Take-away №1:**  
*Mamdani MAX aggregation plus centroid is unforgiving – extra rules must be gated very carefully,
otherwise they “fight” and degrade accuracy.*

---

### 2.3  FIS-2 — **x7 with x2 & x3 as *contextual ties-breakers***

#### 2.3.1  Redesign: *overlap-gated* refinements  
Instead of static “near-boundary” sets, we detect *true ambiguity* mathematically:
$$
\text{gate}_{i,i+1}(x_7) = \min\bigl(\mu_{i}(x_7),\; \mu_{i+1}(x_7)\bigr)
$$

Only when this gate > 0 we let x2/x3 speak.

* **x2, x3 MFs:** three triangles each {Low, Medium, High}.  
* **Five gated rules**: push *upwards* if (x2 OR x3) is High, push *downwards* if Low.

#### 2.3.2  Two new hyper-parameters

| Symbol | Purpose | Typical values |
|--------|---------|----------------|
| `gate_relax` | How much to *widen* the overlap region (adds a small constant before the *min*) | 0.00 – 0.65 |
| `ref_weight` | Multiplier on the firing strength of refinement rules | 1 – 4 |

Both are implemented in `fis2_x7_x2x3.py`; quick sweeps were scripted in
`sweep_gate.sh` and `sweep_grid.sh`.

#### 2.3.3  Empirical findings  

| Setting | run1 Acc | run2 Acc | Explanation |
|---------|---------:|---------:|-------------|
| **gate_relax ≤ 0.25**<br>`ref_weight=3` | 0.54 | **0.54** | Gates too narrow → refinements rarely activate (safe default). |
| **gate_relax = 0.35** | **0.56** | 0.53 | Helps run1 (data happen to have ambiguous 1 | 2 cases) but harms run2 (over-correction). |
| gate_relax ≥ 0.55 | 0.56 | **0.51** | Refines too aggressively; small test becomes unstable. |

**Take-away №2:**  
*In tiny evaluation splits, a single mis-correction can change accuracy by ±2 pp.
Therefore we freeze FIS-2 at a conservative gate, documenting the “sweet spot”
only as a footnote.*

---

### 2.4  FIS-3 — **x7 plus aggregate *S***  

#### 2.4.1  Why aggregate (x1 … x6) into one variable?  
EDA (Section 1) showed:

* Each of x1 … x6 holds ≈ 20 % correlation with the label – *individually weak*.
* They have almost identical distributions.

Averaging them

$$
S = \tfrac{1}{6}\sum_{i=1}^{6} x_i
$$

*keeps the trend* but collapses six dimensions to one → rule count stays 10.

#### 2.4.2  MFs & rules  
* **S:** Low / Medium / High (triangles)  
* **Rule pattern:** identical to FIS-2 but with S in place of {x2,x3}.  
* **Refinement weight sweep** discovered that  
  **`ref_weight = 6`** finally made S influential.

| `ref_weight` | run1 Acc | run2 Acc | Comment |
|-------------:|---------:|---------:|---------|
| 1 … 5 | 0.54 | 0.51 | No visible effect → S still too weak |
| **6** | **0.58** | **0.55** | Corrects many 1 | 2 errors while leaving others intact |
| 10 | 0.57 | 0.55 | Slight “overshoot” – class-3 pushed downwards occasionally |

#### 2.4.3  Boundary re-calibration  
While tuning `ref_weight`, I noticed small shifts of the x7 centroid around
the 3 | 4 | 5 region.  A micro-grid search on the last two boundaries
(28 → 28.5, 34 → 36) removed residual 4↔5 confusion, giving the final matrices in
`results/fis_results/fis3/confusion_*`.

**Take-away №3:**  
*A single, well-weighted secondary feature (S) is more stable than two low-correlated
raw inputs; but its influence must be amplified and the main variable’s
boundaries re-checked.*

---

### 2.5  Common implementation details

| Item | Decision & justification |
|------|--------------------------|
| **Engine** | Plain NumPy; loops unrolled for 100× speed vs naïve Python.  |
| **Defuzzifier** | Centroid (“centre of gravity”) because it produces a *continuous* crisp output that we can later round for classification ↔ same style used in ANFIS regression. |
| **Rounding logic** | `np.clip(np.round(crisp), 1, 5)` – aligns exactly with ANFIS post-processing, ensuring an apples-to-apples comparison later. |
| **Missing rule coverage** | If every μ = 0 (can happen after heavy gating) the engine falls back to crisp = 3 (mid-class) to avoid NaNs. Logged as a warning; occurred < 0.5 % in any split. |

---

### 2.6  What the FIS phase taught me (and shaped the ANFIS phase)

* **Over-fitting danger is reverse!** – too *many* hand-rules *hurt* because
  centroid blends them, unlike data-driven models which over-fit by memorising.
* **Soft gating beats hard “boundary triangles”.**
* **Weak features are not useless** – they just need *either* weight amplification
  **or** dimensionality reduction (the S trick).
* **Evaluation variance on 100 samples** is huge: ±0.02 in accuracy by moving a single point.
  ⇒ Large-scale (>50 k) ANFIS splits are mandatory for a trustworthy metric.
* Custom code gave the microscopic control required to experiment with ideas
  like `gate_relax` and `ref_weight`; such knobs are impossible in
  off-the-shelf libraries.

These insights directly motivated the *trainable* Takagi–Sugeno systems in Section 3,
where premises and thresholds are **learned** instead of manually nudged.



## 3 · Adaptive Neuro-Fuzzy Inference System (ANFIS)

This section documents (i) the ANFIS architecture we implemented from scratch, (ii) the exact training/tuning pipeline we followed (and why), and (iii) the complete set of ANFIS results that exist in the submission package (no placeholder figures/tables).

All artefacts referenced below exist under `anfis_results/` and `src/` in the submission package.  
We do not introduce any new plots or results beyond what is already saved there.

---

### 3.1 Problem framing: classification via a regression-style ANFIS output

ANFIS (Takagi–Sugeno type) naturally produces a continuous output $\hat{y}$.  
Our target is 5 discrete classes: $\text{remarks} \in \{1,2,3,4,5\}$.

So we solved classification by:

- Training ANFIS to predict a continuous $\hat{y}$ close to the numeric labels  
  $\{1,2,3,4,5\}$ (MSE objective).

- Converting $\hat{y}$ to a class prediction using 4 learned thresholds:

$$
\hat{c} =
\begin{cases}
1 & \hat{y} < t_1 \\
2 & t_1 \le \hat{y} < t_2 \\
3 & t_2 \le \hat{y} < t_3 \\
4 & t_3 \le \hat{y} < t_4 \\
5 & \hat{y} \ge t_4
\end{cases}
$$

- Optimizing thresholds to maximize training accuracy (details in §3.6).

This is exactly what our code does (`threshold search + apply_thr`) and why all ANFIS result folders store thresholds inside `metrics.json`.

---

### 3.2 ANFIS-specific dataset splits (balanced + two iterations)

ANFIS experiments used dedicated, balanced splits generated by:

- Script: `src/make_anfis_splits.py`  
- Output folder: `data/anfis_splits/`

The split strategy (as implemented in the script):

- Sample a balanced subset per class (same number from each `remarks` label).
- Create a stratified train/test split.
- Repeat for two independent iterations (**Iter-1** and **Iter-2**) to verify stability.

This is why ANFIS results are stored as:

anfis_results/iter1/...
anfis_results/iter2/...


---

### 3.3 Preprocessing for ANFIS: Min-Max scaling (and why it matters here)

ANFIS training involves two coupled mechanisms:

- Least squares / ridge regression for consequent parameters  
- SGD-style gradient updates for premise parameters (MF centers/widths)

If inputs have very different numeric ranges, then:

- Gaussian MF activations can saturate (too wide / too narrow effect).
- Premise gradients can be poorly conditioned (updates dominated by the largest-scale feature).
- Consequent LS design matrix becomes less stable numerically.

Therefore we applied **Min-Max scaling** in ANFIS runs and saved the scaler bounds in each run’s config.

Example (7D run):  
`anfis_results/iter1/anfis_7d_lr0p003/config.json`

    "scaler": { "lo": [...], "hi": [...] }


Example (S+x7 run):  
`anfis_results/iter1/anfis_Sx7/config.json`

So the report can reproduce exactly which scaling ranges were used, without guessing.

---

### 3.4 ANFIS architecture used: first-order Takagi–Sugeno with Gaussian MFs

All ANFIS models in this project follow the same core structure.

#### 3.4.1 Membership functions (premise layer)

For each input dimension $d$, we define $K_d$ Gaussian membership functions:

$$
\mu_{d,k}(x_d) = \exp\!\left(-\frac{(x_d - c_{d,k})^2}{2\sigma_{d,k}^2}\right)
$$

Centers $c_{d,k}$ are initialized by quantiles of the training data  
(see `init_mfs_quantiles` in `src/anfis_models/anfis_generic.py` and `anfis_x7.py`).

Widths $\sigma_{d,k}$ are initialized from spacing between neighboring centers, multiplied by `sigma_mult`.

This initialization:

- covers the observed input range robustly,
- avoids empty/unused MFs on skewed distributions,
- provides a stable starting point for ridge + gradient training.

#### 3.4.2 Rule layer (product inference)

A rule corresponds to selecting one MF index from each dimension:

$$
R_m: \text{IF } x_1 \text{ is } A_{1,i_1} \wedge \cdots \wedge x_D \text{ is } A_{D,i_D} \Rightarrow f_m(x)
$$

Rule firing strength:

$$
w_m(x) = \prod_{d=1}^{D} \mu_{d,i_d}(x_d)
$$

Normalization:

$$
\bar{w}_m(x) = \frac{w_m(x)}{\sum_j w_j(x) + \varepsilon}
$$

This is exactly how firing strengths are computed in the `forward(...)` function.

#### 3.4.3 Consequent layer (first-order / linear)

Each rule has a first-order linear consequent:

$$
f_m(x) = a_m^\top x + b_m
$$

Total output:

$$
\hat{y}(x) = \sum_m \bar{w}_m(x)\, f_m(x)
$$

---

### 3.5 Why we implemented ANFIS ourselves (and where in the code)

We did not call a black-box ANFIS library.  
Instead, the ANFIS implementation is explicitly coded so we can control:

- MF initialization by quantiles  
- product inference + normalization  
- ridge regression consequent solving  
- premise gradient update rules  
- threshold optimization  
- logging of training dynamics

Files:

- Generic multi-dimensional ANFIS: `src/anfis_models/anfis_generic.py`
- Single-input ANFIS baseline: `src/anfis_models/anfis_x7.py`
- S + x7 wrapper: `src/anfis_models/anfis_sx7.py`

This makes the results fully verifiable: every run stores  
`config.json`, `train_history.json`, `metrics.json`, and a formatted confusion matrix.

---

### 3.6 Training algorithm: hybrid learning + threshold search

Training follows a hybrid loop (per epoch) implemented in `train_generic(...)`.

**Step A — Forward pass**

Compute MF activations $\mu$, rule strengths $w$, normalized $\bar{w}$, and ANFIS output $\hat{y}$.

**Step B — Consequent update (ridge regression)**

$$
\theta = \arg\min_\theta \|X_{\text{design}}\theta - y\|^2 + \lambda \|\theta\|^2
$$

Implemented as `solve_consequents_ridge(...)`.  
`ridge = 0.001` is stored in each run’s config.

**Step C — Threshold optimization**

Thresholds $t_1..t_4$ are optimized to maximize training accuracy.

- `thr_mode = "optimize"`
- `thr_refresh = 5`

This aligns classification with the learned $\hat{y}$ distribution.

**Step D — Premise update (SGD)**

Update $c$ and $\sigma$ using learning rate `lr`.  
Log `delta_c`, `delta_s` into `train_history.json`.

---

### 3.7 The ANFIS model variants we evaluated

#### 3.7.1 ANFIS-x7 (baseline)

- 5 Gaussian MFs on x7 → 5 rules  
- Accuracy ≈ 0.51–0.52

#### 3.7.2 ANFIS-7D

- MF counts: `[2,2,2,2,2,2,5]`
- Rule count: $2^6 \times 5 = 320$
- Consequent params ≈ $320 \times 8 = 2560$
- Best accuracy ≈ **0.993**

#### 3.7.3 ANFIS-(S + x7)

Aggregate feature:

$$
S = \tfrac{1}{6} \sum_{i=1}^{6} x_i
$$

- MF counts: `S=3`, `x7=5`
- Rules: $3 \times 5 = 15$
- Consequent params ≈ $15 \times 3 = 45$
- Accuracy ≈ **0.991–0.992**

---

### 3.8 Hyperparameters we tuned

- `lr`, `ridge`, `sigma_mult`
- `thr_mode`, `thr_refresh`
- MF counts per input

Final settings saved in `anfis_results/`.

---

### 3.9 ANFIS results (from saved artefacts)

| Model | Iter | Folder | Best epoch | Best test acc | Best MSE | Best thresholds |
|------|------|--------|-----------|---------------|----------|----------------|
| ANFIS-x7 | 1 | anfis_x7_patch | 3 | 0.51416 | 0.55890 | [1.55115, 2.49593, 3.54743, 4.54000] |
| ANFIS-x7 | 1 | anfis_x7_tune1 | 3 | 0.51656 | 0.55930 | [1.67134, 2.51106, 3.56011, 4.07138] |
| ANFIS-7D | 1 | anfis_7d_lr0p003 | 25 | 0.99296 | 0.06571 | [1.57590, 2.52079, 3.45805, 4.46051] |
| ANFIS-7D | 2 | anfis_7d_lr0p003 | 60 | 0.99280 | 0.06575 | [1.57673, 2.52142, 3.46524, 4.45547] |
| ANFIS-(S+x7) | 1 | anfis_Sx7 | 80 | 0.99216 | 0.06905 | [1.52324, 2.48741, 3.54191, 4.53305] |
| ANFIS-(S+x7) | 2 | anfis_Sx7 | 98 | 0.99088 | 0.06963 | [1.52400, 2.48834, 3.53408, 4.54638] |

---

### 3.10 Confusion matrices

Each run stores a readable confusion matrix as: confusion_matrix_best_pretty.txt
Below are the existing matrices from the best epoch outputs.

#### 3.10.1 ANFIS-7D (Iter-1) — anfis_results/iter1/anfis_7d_lr0p003/confusion_matrix_best_pretty.txt

| true \ pred |   1  |   2  |   3  |   4  |   5  |
|-------------|------|------|------|------|------|
| **1**       | 2491 |   7  |   1  |   0  |   1  |
| **2**       |  11  | 2477 |  12  |   0  |   0  |
| **3**       |   2  |  14  | 2466 |  18  |   0  |
| **4**       |   0  |   0  |  11  | 2479 |  10  |
| **5**       |   0  |   0  |   0  |   1  | 2499 |

This is near-perfect across all classes, including class 5.

#### 3.10.2 ANFIS-(S+x7) (Iter-1) — anfis_results/iter1/anfis_Sx7/confusion_matrix_best_pretty.txt

| true \ pred |   1  |   2  |   3  |   4  |   5  |
|-------------|------|------|------|------|------|
| **1**       | 2483 |  17  |   0  |   0  |   0  |
| **2**       |  10  | 2444 |  46  |   0  |   0  |
| **3**       |   0  |  25  | 2432 |  43  |   0  |
| **4**       |   0  |   0  |  20  | 2468 |  12  |
| **5**       |   0  |   0  |   0  |  25  | 2475 |

S+x7 remains extremely strong with a much smaller rule base; most errors occur between adjacent classes (2↔3, 3↔4, 4↔5), which matches the ordinal structure of the target.

#### 3.10.3 ANFIS-x7 baseline — anfis_results/iter1/anfis_x7_patch/confusion_matrix_best_pretty.txt

| true \ pred |   1  |   2  |   3  |   4  |   5  |
|-------------|------|------|------|------|------|
| **1**       | 1489 |  911 |  98  |   2  |   0  |
| **2**       |  411 | 1252 |  807 |  30  |   0  |
| **3**       |   0  |  435 | 1011 | 1054 |   0  |
| **4**       |   0  |   0  |   5  | 1466 | 1029 |
| **5**       |   0  |   0  |   0  |  712 | 1788 |

This shows the baseline collapses into broad overlaps between adjacent classes, especially in the middle (2–4).

### 3.11 Discussion
We can now explain the performance gap using only what is evidenced by the saved designs and results.

#### 3.11.1 x7-only ANFIS is under-expressive for the “middle classes” structure

Even though **x7** is strongly correlated with the target (from EDA), the confusion matrix shows:

- **Class 1** often becomes **Class 2**  
- **Class 2** spreads heavily into **Class 3**  
- **Class 3** spreads heavily into **Class 4**  
- **Class 4** spreads into **Class 5**

So the model behaves like a **coarse ordinal regressor** without enough capacity to create clean, class-separating regions along **x7**.

With **x7 alone**:

- **5 Gaussian MFs** → only **5 localized rule regions** on a single axis  
- Each rule has only a **1D linear consequent**  
  $$
  a x_7 + b
  $$

That setup struggles if:

- class boundaries are not sharply separable by **x7** alone (overlap / noise),  
- class transition widths are uneven (some boundaries “tighter” than others),  
- the correct mapping requires additional context beyond **x7**.

The results confirm this: **best accuracy stays around 0.51–0.52** in both baseline runs.

#### 3.11.2 Adding x1…x6 (7D) gives the model extra “context” to resolve ambiguity

**ANFIS-7D** uses:

- coarse partitions for **x1…x6** (**2 MFs each**),  
- detailed partition for **x7** (**5 MFs**),  
- **total 320 rules**.

This means the model can express rules like:

> “x7 suggests class 3, but if x1..x6 pattern indicates stronger/weaker performance, shift toward class 2/4”

Even if each of **x1..x6** is weak individually, the **7D rule base** allows conditional corrections around ambiguous **x7** regions, producing the observed near-perfect confusion matrix.

This is exactly what we see: the biggest gains show up in the **“middle” classes (2–4)**, where **x7-only failed**.

---

#### 3.11.3 S+x7 works because it captures the “shared signal” in x1…x6 without rule explosion

**ANFIS-(S + x7)** compresses **x1..x6** into **S**:

- still supplies extra information beyond **x7**,  
- avoids **320 rules** and uses only **15 rules**.

The confusion matrix shows errors are mostly between **adjacent classes** (ordinally “reasonable”), and accuracy remains **~0.991–0.992**.

So **S + x7** is the **“sweet spot”**:

- minimal rule base,  
- strong generalization,  
- interpretable **2D fuzzy partition**.

#### 3.11.4 Why threshold optimization and refresh mattered  

In every strong run, best thresholds are **not trivial evenly spaced values**; they are **learned** and slightly adjusted between **iter-1** and **iter-2**.

Because  
$$
\hat{y}
$$  
is produced by a **regression-style TS output**, the absolute **scale / offset** of  
$$
\hat{y}
$$  
is **model-dependent**. Threshold optimization ensures classification is aligned to the learned  
$$
\hat{y}
$$  
distribution rather than assuming fixed cutoffs.

This is why we **store and report thresholds** in `metrics.json` for each run.

---